T4 settings vs the local 4GB defaults: **batch_size 32, grad_accum_steps 2** — same effective batch of 64, roughly half the activation memory. The T4 reports ~14.6GiB *usable* VRAM, so batch 64 in one pass OOMs (the VRAM guard will warn if you try). Model architecture and all seeds/schedules are unchanged.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# 1. Confirm the GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 2. Get the code + dependencies

Colab already has a CUDA build of torch — we only install the missing packages (do **not** reinstall torch here; `requirements.txt`'s cu130 pin is for the local Windows machine).


In [ ]:
!git clone https://github.com/Bit-Sahil04/toy-pixel-diffuser.git

%cd toy-pixel-diffuser

!pip install -q datasets huggingface_hub pillow numpy tqdm matplotlib


In [ ]:
# 3. Environment sanity check (CUDA + AMP on the T4)

!python check_env.py


## 4. (Recommended) Mount Google Drive for checkpoints

Colab disks are wiped when the runtime dies. This saves checkpoints/sample grids/logs to Drive so a disconnected session loses at most one checkpoint interval.

Skip this cell to train fully ephemeral.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/pixel_diffuser/run1'

DRIVE_SAMPLES = '/content/drive/MyDrive/pixel_diffuser/run1_samples'

DRIVE_LOG = '/content/drive/MyDrive/pixel_diffuser/run1_log.csv'


## 5. Download + inspect the dataset

~300MB (LPC 4-view sprites, 50k images @ 128x128 RGBA). Prints real dims/modes and a sample caption; asserts no flip/rotation transforms.


In [ ]:
!python data.py


## 6. (Optional) Quick smoke test first

60 steps on 512 images to confirm the T4 setup end-to-end before the real run (~2 min, most of it the fixed-seed sample grid).


In [ ]:
# 6. (Optional) Quick smoke test first
# 60 steps on 512 images to confirm the T4 setup end-to-end before the real
# run (~7 min on T4: ~1 min training + one fixed-seed sample grid).
# Isolated log: re-running this cell APPENDS to the CSV; cell 1d's median-
# delta math tolerates that, but never point the smoke run at the real log.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train.py --limit 512 --max_train_steps 60 --batch_size 32 --grad_accum_steps 2 --log_csv logs/smoke_log.csv


## 7. Pre-flight checklist - REQUIRED before training

Automated version of `docs/preflight_checklist.md`. Run the four cells below in order and
fix any FAIL before continuing. ~10-15 min total (most of it is the tiny-overfit sample pass
and the resume-test grid).

| cell | check | pass condition |
|---|---|---|
| 1a | tiny-overfit on 16 REAL sprites | samples reproduce targets, mean NCC > 0.6 |
| 1b | scheduler math + **visual** noise strip | closed form + diffusers AGREE; image dissolves at the right rate |
| 1c | config-effect summary | attention/prediction/EMA/batch actually as intended |
| 1d | throughput, ETA, resume integrity | GPU util high, no loss spike after resume |

**Why:** loss going down proves the training loop works - and proves nothing about the
sampler, the config, or anything downstream. We once had a healthy loss curve while the
sampler's posterior math was wrong; every grid it produced was degraded. These cells give
you a second, independent, cheap signal before you commit GPU-hours.

In [ ]:
# 0a - Shared flags: single source of truth for the T4 run.
# Section 8 trains with exactly these; cell 1c validates them; cell 1d projects
# hours from them. To change training behavior, change THIS cell only.
CFG_FLAGS = dict(batch_size=32, grad_accum_steps=2, sample_every=1000,
                 max_train_steps=20000)
print('shared flags:', CFG_FLAGS)

In [ ]:
# 1a - Tiny-overfit test (checklist #1; highest signal per minute spent)
# TWO STAGES, because a single end-to-end NCC can false-alarm:
#   Stage A - memorization probe: one forward pass at t=300/500, recover x0 from
#             x_t. No 1000-step drift. Untrained model scores ~0.2-0.5, overfit
#             model >0.9. (Low-t probes are MEANINGLESS: at t=50 x_t is ~99%
#             signal, so even an untrained model "reconstructs" it.)
#   Stage B - end-to-end: full 1000-step DDPM sample from pure noise. NOTE:
#             sampling does NOT preserve slot order - sample i is SOME training
#             image, not target i - so the metric is BEST-MATCH NCC (each sample
#             vs its nearest of the N targets), never positional pairing.
# 8 REAL sprites (faster memorization than 16), early-stop at loss<0.02, cap 1500.
# PASS = reconNCC(t=500) > 0.8 AND mean best-match sample NCC > 0.5.
import torch
import matplotlib.pyplot as plt
from config import Config
from data import PixelArtDataset, ensure_data, denormalize
from diffusion import Diffusion
from model import build_model
from train import set_seed

N = 8
MAX_STEPS = 1500          # ~5-8 min on T4; usually early-stops well before this
EARLY_STOP_LOSS = 0.02
LR = 2e-4                 # same lr as the real run - the test exercises the same setup
set_seed(0)
cfg = Config(); cfg.batch_size = N; cfg.grad_accum_steps = 1
device = torch.device('cuda')
images_dir, captions_csv = ensure_data()
ds = PixelArtDataset(images_dir, captions_csv, image_size=cfg.image_size,
                     unconditional=True, limit=N)
# shuffle=False: the SAME fixed 8 sprites every run, so NCC is comparable
x0 = next(iter(torch.utils.data.DataLoader(ds, batch_size=N)))['image']
x0 = x0.to(device).to(memory_format=torch.channels_last)
x0_rgb = denormalize(x0).cpu()

model = build_model(cfg).to(device).to(memory_format=torch.channels_last)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler('cuda')
diff = Diffusion(cfg.timesteps, cfg.beta_schedule, cfg.prediction_target, device)

def ncc(a, b):  # normalized cross-correlation; a (n,3,H,W), b (m,3,H,W) -> (n,m)
    a = a.flatten(1); a = a - a.mean(1, keepdim=True)
    b = b.flatten(1); b = b - b.mean(1, keepdim=True)
    return (a @ b.T) / (a.norm(dim=1)[:, None] * b.norm(dim=1)[None, :] + 1e-8)

def recon_ncc(t_eval, gen_seed=11):
    """Stage-A probe: one forward pass at fixed t - can the model recover x0
    from x_t? Direct memorization measure, no sampler drift."""
    with torch.no_grad():
        g = torch.Generator().manual_seed(gen_seed)
        eps = torch.randn(x0.shape, generator=g).to(device)
        t = torch.full((N,), t_eval, device=device, dtype=torch.long)
        x_t = diff.q_sample(x0, t, eps)
        pred_x0 = diff.predict_x0(x_t, t, model(x_t, t)).clamp(-1, 1)
    return ncc(denormalize(pred_x0).cpu(), x0_rgb).diagonal().mean().item()

losses = []
model.train()
for step in range(1, MAX_STEPS + 1):
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda'):
        loss = diff.training_losses(model, x0)
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    losses.append(loss.item())
    if step == 1 or step % 100 == 0:
        print(f'overfit step {step:4d} | loss {loss.item():.4f} '
              f'| reconNCC(t=300) {recon_ncc(300):.3f}')
    if loss.item() < EARLY_STOP_LOSS:
        print(f'early stop at step {step}: loss {loss.item():.4f} < {EARLY_STOP_LOSS}')
        break

final_loss = losses[-1]
assert final_loss < losses[0] * 0.2, 'loss did not fall - fix training path before sampling'

# ---- Stage A: memorization (decisive signal) ----
r300, r500 = recon_ncc(300), recon_ncc(500)
print(f'Stage A - memorization: reconNCC(t=300) = {r300:.3f} | reconNCC(t=500) = {r500:.3f}')
if r500 < 0.8:
    if final_loss > 3 * EARLY_STOP_LOSS:
        print('FAIL -> UNDERTRAINED: loss was still falling. Raise MAX_STEPS and re-run '
              '(do NOT start a real run yet).')
    else:
        print('FAIL -> STRUCTURAL BUG: loss converged but the model cannot reproduce its '
              'own training data. Investigate model/data/loss. STOP.')
    raise AssertionError(f'reconNCC(t=500)={r500:.3f} < 0.8')

# ---- Stage B: end-to-end sampler (BEST-MATCH metric) ----
samples = diff.sample(model, N, cfg.image_size, seed=0).cpu()
S = ncc(samples, x0_rgb)                    # (n_samples, n_targets)
best, match = S.max(dim=1)                  # each sample's closest training sprite
mean_ncc = best.mean().item()
print('best-match sample NCC (sample -> matched target):',
      ' '.join(f'{b:.2f}(->t{m.item()})' for b, m in zip(best, match)))
print(f'Stage B - full 1000-step DDPM sample: mean best-match NCC = {mean_ncc:.3f} -> '
      + ('PASS' if mean_ncc > 0.5 else
         'FAIL - samples resemble NO training image while memorization passed: '
         'sampler bug - STOP'))
assert mean_ncc > 0.5, 'best-match sample NCC too low while memorization passed - sampler bug'

fig, axes = plt.subplots(2, N, figsize=(N * 1.35, 3.1))
for i in range(N):
    axes[0, i].imshow(x0_rgb[match[i]].permute(1, 2, 0).clamp(0, 1))
    axes[0, i].set_title(f'target {match[i].item()}', fontsize=7)
    axes[0, i].axis('off')
    axes[1, i].imshow(samples[i].permute(1, 2, 0).clamp(0, 1)); axes[1, i].axis('off')
plt.suptitle(f'tiny-overfit: nearest target (top) vs 1000-step DDPM samples (bottom) '
             f'- best-match NCC {mean_ncc:.3f}')
plt.show()

plt.figure(figsize=(5, 2.5))
plt.plot(losses); plt.title('overfit loss (should flatten)'); plt.xlabel('step'); plt.show()

In [ ]:
# 1b - Scheduler numerical + VISUAL check (checklist #2)
# Independent confirmations of diffusion.py's schedule and posterior math:
#   (1) closed-form cosine recomputed HERE, not imported,
#   (2) cross-check against diffusers' trusted DDPMScheduler,
#   (3) algebra identities: the posterior-mean coefficient form vs its closed
#       form, and a predict_x0 ROUND-TRIP for BOTH parameterizations (feed the
#       true noise/target -> x0 must come back; this is the exact function that
#       held the original sampler bug, and the spot table alone can't catch it),
#   (4) a printed spot-table of the exact sampler coefficients,
#   (5) a forward-noise strip so you SEE the schedule destroy a real sprite.
import math
import torch
import matplotlib.pyplot as plt
from config import Config
from data import PixelArtDataset, ensure_data, denormalize
from diffusion import Diffusion

cfg = Config()
print(f'checking: timesteps={cfg.timesteps} schedule={cfg.beta_schedule} '
      f'target={cfg.prediction_target}')
diff = Diffusion(cfg.timesteps, cfg.beta_schedule, cfg.prediction_target, 'cpu')
T = diff.timesteps

if cfg.beta_schedule == 'cosine':
    # (1) independent closed form (Nichol & Dhariwal 2021 cosine).
    # Apply the SAME documented clamp as diffusion.py (betas in [1e-4, 0.999]) —
    # the lower clamp binds for the first ~30 steps (unclamped beta_0 = 4.1e-5)
    # and shifts abar by <=0.2%; that is deliberate, not an error.
    steps = torch.arange(T + 1, dtype=torch.float64)
    abar_cf = torch.cos(((steps / T) + 0.008) / 1.008 * math.pi / 2) ** 2
    abar_cf = abar_cf / abar_cf[0]
    betas_cf = (1 - (abar_cf[1:] / abar_cf[:-1])).clamp(1e-4, 0.999)
    abar_cf = torch.cumprod(1 - betas_cf, dim=0)
    ok = torch.allclose(diff.alphas_cumprod, abar_cf.float(), atol=1e-5)
    print(f'(1) abar vs closed form (all {T} t, same clamp): '
          f"{'AGREE' if ok else 'DISAGREE - DO NOT TRAIN'}")
    assert ok

    # (2) trusted-reference diff
    try:
        from diffusers import DDPMScheduler
        ref = DDPMScheduler(num_train_timesteps=T, beta_schedule='cosine')
        dmax = (ref.betas - diff.betas).abs().max().item()
        agree = torch.allclose(ref.betas, diff.betas, atol=1e-4)
        print(f'(2) betas vs diffusers DDPMScheduler(cosine): '
              f"{'AGREE' if agree else 'DISAGREE'} (max diff {dmax:.2e})")
        # diffusers does NOT lower-clamp, so beta_0 differs slightly (6e-5);
        # atol=1e-4 tolerates exactly that documented difference
    except ImportError:
        print('(2) SKIP: diffusers not installed - %pip install diffusers, then re-run')
else:
    print('(1)/(2) SKIP: closed form + diffusers cross-check are cosine-specific; '
          'add the closed form for your schedule here before trusting it')

# (3) algebra identities - schedule-agnostic, catch sign/coefficient bugs prints can't
print('(3) algebra identities:')
for i in (1, 100, 500, 999):
    abar, abar_p = diff.alphas_cumprod[i], diff.alphas_cumprod[i - 1]
    bt, at = diff.betas[i], 1.0 - diff.betas[i]
    den = 1.0 - abar
    x0 = torch.randn(1, 3, 16, 16)
    eps = torch.randn(1, 3, 16, 16)
    x_t = abar.sqrt() * x0 + (1 - abar).sqrt() * eps
    mu_coef = (abar_p.sqrt() * bt / den) * x0 + (at.sqrt() * (1 - abar_p) / den) * x_t
    mu_closed = (x_t - bt / (1 - abar).sqrt() * eps) / at.sqrt()
    err = (mu_coef - mu_closed).abs().max().item()
    assert err < 1e-4, f'posterior-mean identity failed at t={i}: {err:.2e}'
print('    posterior mean: coef form == closed form at t={1,100,500,999} (err < 1e-4) OK')

for target in ('epsilon', 'v_prediction'):
    d_chk = Diffusion(cfg.timesteps, cfg.beta_schedule, target, 'cpu')
    worst = 0.0
    # t>950 excluded: sqrt(abar)<1e-4 there amplifies float32 rounding ~5000x;
    # the identity still holds, but the CHECK would false-fail on precision alone
    for t_ in (1, 50, 300, 500, 900):
        x0 = torch.randn(2, 3, 32, 32)
        eps = torch.randn(2, 3, 32, 32)
        t = torch.full((2,), t_, dtype=torch.long)
        x_t = d_chk.q_sample(x0, t, eps)
        if target == 'epsilon':
            out = eps
        else:
            ac = d_chk.sqrt_alphas_cumprod[t][:, None, None, None]
            om = d_chk.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
            out = ac * eps - om * x0
        worst = max(worst, (d_chk.predict_x0(x_t, t, out) - x0).abs().max().item())
    assert worst < 1e-5, f'predict_x0 round-trip failed for {target}: {worst:.2e}'
    print(f'    predict_x0 round-trip ({target:13s}): max err t={{1,50,300,500,900}} '
          f'= {worst:.2e} OK')

# (4) printed spot-table: the exact values the sampler will use
print(f"{'t':>5} {'beta_t':>9} {'abar_t':>9} {'abar_prev':>10} "
      f"{'coef_x0':>9} {'coef_xt':>9} {'post_var':>9}")
for i in (1, 100, 500, 999):
    abar, abar_p = diff.alphas_cumprod[i], diff.alphas_cumprod[i - 1]
    bt, at = diff.betas[i], 1.0 - diff.betas[i]
    den = 1.0 - abar
    cx0 = (abar_p.sqrt() * bt / den).item()
    cxt = (at.sqrt() * (1.0 - abar_p) / den).item()
    var = (bt * (1.0 - abar_p) / den).item()
    print(f'{i:5d} {bt.item():9.5f} {abar.item():9.5f} {abar_p.item():10.5f} '
          f'{cx0:9.5f} {cxt:9.5f} {var:9.6f}')

# (5) VISUAL: schedule curves + forward-noise strip on a REAL sprite
abar_prev = torch.cat([torch.ones(1), diff.alphas_cumprod[:-1]])
post_var = diff.betas * (1 - abar_prev) / (1 - diff.alphas_cumprod)
fig, ax = plt.subplots(1, 3, figsize=(14, 3))
ax[0].plot(diff.betas); ax[0].set_title('beta_t (per-step noise)')
ax[1].plot(diff.sqrt_alphas_cumprod, label='sqrt(abar): signal kept')
ax[1].plot(diff.sqrt_one_minus_alphas_cumprod, label='sqrt(1-abar): noise added')
ax[1].legend(); ax[1].set_title('signal vs noise over t')
ax[2].plot(post_var); ax[2].set_title('posterior variance')
plt.tight_layout(); plt.show()

ds = PixelArtDataset(*ensure_data(), image_size=128, unconditional=True, limit=8)
img = ds[0]['image'][None]                    # one REAL sprite in [-1,1]
ts = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]
eps = torch.randn(img.shape, generator=torch.Generator().manual_seed(3))  # ONE eps for all t
fig, axes = plt.subplots(1, len(ts), figsize=(len(ts) * 1.3, 1.8))
for a, t in zip(axes, ts):
    xt = diff.q_sample(img, torch.tensor([t]), eps)
    a.imshow(denormalize(xt)[0].permute(1, 2, 0).clamp(0, 1))
    a.set_title(f't={t}', fontsize=8); a.axis('off')
plt.suptitle('q_sample on ONE real sprite, same eps each t: intact to ~t=400, '
             'ghost ~t=600, pure static by ~t=800')
plt.show()

In [ ]:
# 1c - Config sanity: what the config ACTUALLY does (checklist #3)
# Validates the CFG_FLAGS cell (0a) - the exact flags section 8 will train with,
# not notebook defaults. Classic silent bug this catches:
# attention_resolutions=(16, 8) never matching real feature maps [128, 64, 32].
import torch
from config import Config
from model import SelfAttention, build_model
from train import print_config_summary

cfg = Config()
for k, v in CFG_FLAGS.items():          # mirror the shared-flags cell
    assert hasattr(cfg, k), f'CFG_FLAGS key {k} is not a Config field'
    setattr(cfg, k, v)
model = build_model(cfg)
print_config_summary(cfg, model)        # same summary train.py prints at startup

feat_sizes = [cfg.image_size // 2 ** i for i in range(len(cfg.channel_mults))]
in_cfg = [s for s in feat_sizes if s in cfg.attention_resolutions]
le_cfg = [s for s in feat_sizes if s <= max(cfg.attention_resolutions)]
print(f'feature sizes: {feat_sizes} | attention_resolutions={cfg.attention_resolutions}')
print(f'  exact-match semantics (`in`): would fire at {in_cfg or "NOTHING"}  <- the old silent bug')
print(f'  <= semantics (actual):        fires at {le_cfg} (+ the always-on bottleneck mid)')

# runtime verification, not just prints: count what the module tree REALLY built
n_attn = sum(1 for m in model.modules() if isinstance(m, SelfAttention))
expected = 1 + 2 * len(le_cfg)   # mid + down/up at each qualifying level
assert n_attn == expected, f'attention modules {n_attn} != expected {expected}'
n_params = sum(p.numel() for p in model.parameters())
# loose sanity band around the known ~14.98M - catches accidental width/depth drift
assert 14e6 < n_params < 16e6, (f'param count {n_params / 1e6:.2f}M outside the '
                                '14-16M band - config drifted?')
print(f'PASS: {n_attn} attention modules (= {expected} expected), '
      f'{n_params / 1e6:.2f}M params, flags match CFG_FLAGS')

In [ ]:
# 1d - Throughput, ETA, data integrity, resume (checklist #4, #5, #6, #7)
import csv
import statistics
import subprocess
import threading
import time
from pathlib import Path

assert 'CFG_FLAGS' in globals(), 'run the 0a shared-flags cell first'
eff = CFG_FLAGS['batch_size'] * CFG_FLAGS['grad_accum_steps']
max_steps, se = CFG_FLAGS['max_train_steps'], CFG_FLAGS['sample_every']
print(f'flags from 0a: batch {CFG_FLAGS["batch_size"]} x accum '
      f'{CFG_FLAGS["grad_accum_steps"]} = {eff} | {max_steps} steps | grid every {se}')
print(f'{max_steps} steps ~ {max_steps * eff / 50000:.0f} epochs over 50k images\n')

print('GPU right now (idle is fine):')
print(subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)

# --- data integrity: decode EVERY image through the REAL preprocessing path ---
# The smoke run touches <=2k of 50k files. One corrupt PNG mid-dataset would
# otherwise kill the real run hours in. ~2-4 min on T4 with 2 workers.
RUN_DECODE_SWEEP = True
if RUN_DECODE_SWEEP:
    from torch.utils.data import DataLoader
    from data import PixelArtDataset, ensure_data
    t0 = time.time()
    ds_full = PixelArtDataset(*ensure_data(), image_size=128, unconditional=True)
    n = 0
    for batch in DataLoader(ds_full, batch_size=256, num_workers=2):
        n += batch['image'].shape[0]
    print(f'decode sweep: {n} images through the real pipeline in {time.time() - t0:.0f}s '
          f'- a corrupt file would have raised HERE, not 6h into training\n')

# --- throughput: MEDIAN of per-row elapsed deltas from the isolated smoke log ---
# (mean-over-steps is poisoned by the smoke grid: ~360s of sampling inside a 60-step
# run inflated the estimate ~16x; the median ignores that single outlier, and
# re-running the smoke cell can't corrupt this - it appends, median doesn't care)
log_path = Path('logs/smoke_log.csv')
if log_path.exists():
    rows = list(csv.reader(open(log_path)))[1:]
if not log_path.exists() or len(rows) < 10:
    print('run the smoke-test cell first for a throughput estimate')
else:
    el = [float(r[3]) for r in rows]
    deltas = sorted(b - a for a, b in zip(el, el[1:]))
    sps = deltas[len(deltas) // 2]
    train_h = sps * max_steps / 3600
    grids = max_steps // se + 1
    grid_h = grids * 12 / 60                  # ~12 min per 4x4 grid on T4
    print(f'measured {sps:.2f} s/step (median of {len(deltas)} deltas, grid pause '
          'excluded) -> training ~' + f'{train_h:.1f} h')
    print(f'grids: {grids} x ~12 min at --sample_every {se} = ~{grid_h:.1f} h')
    print(f'BUDGET ~{train_h + grid_h:.1f} h total. Decide now what 25/50/100% progress '
          'looks like (docs/preflight_checklist.md #7).\n')

# --- resume integrity on an EPHEMERAL dir (never touches Drive checkpoints) ---
# Each run emits one forced sample grid at its first step (by design): the first
# is checklist #4's "look at an undertrained grid" (blurry blobs, NOT noise).
# Budget ~12-13 min for this block on T4 (2 grids x ~6 min).
env = 'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True '
tmp = Path('/content/preflight_resume'); (tmp / 's').mkdir(parents=True, exist_ok=True)
common = (f'--limit 512 --batch_size {CFG_FLAGS["batch_size"]} '
          f'--grad_accum_steps {CFG_FLAGS["grad_accum_steps"]} --sample_every 100000 '
          f'--checkpoint_dir {tmp} --samples_dir {tmp}/s --log_csv {tmp}/log.csv')

util, stop = [], threading.Event()
def poll():
    while not stop.is_set():
        r = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu',
                            '--format=csv,noheader,nounits'], capture_output=True, text=True)
        if r.returncode == 0 and r.stdout.strip().isdigit():
            util.append(int(r.stdout.strip()))
        stop.wait(2.0)

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-1000:])
    assert r.returncode == 0, r.stderr[-1500:]

th = threading.Thread(target=poll, daemon=True); th.start()
run(env + f'python train.py --max_train_steps 40 --ckpt_every 40 {common}')
run(env + f'python train.py --max_train_steps 42 --resume_from {tmp}/latest.pt {common}')
stop.set(); th.join(timeout=5)
if util:
    med = statistics.median(util)
    print(f'GPU util during those runs: median {med:.0f}%, max {max(util)}% '
          f'({len(util)} samples)')
    if med < 50:
        print('WARN: utilization low -> I/O or CPU bound; see checklist #5 before '
              'the real run (Drive writes, num_workers, per-step syncs)')
else:
    print('could not sample nvidia-smi - watch utilization manually in the real run')

# --- loss continuity: MEAN of last 3 pre-resume steps vs the 2 post-resume steps ---
# (single-step losses swing +-50%; means + a 2x threshold kill the false WARN)
log = list(csv.reader(open(tmp / 'log.csv')))[1:]
pre = sum(float(r[1]) for r in log[-5:-2]) / 3
post = sum(float(r[1]) for r in log[-2:]) / 2
print(f'mean loss: last 3 pre-resume steps {pre:.4f} -> 2 post-resume steps {post:.4f}')
if post < pre * 2:
    print('PASS: resume continues smoothly (optimizer + AMP + EMA state restored)')
else:
    print('WARN: loss jumped after resume - optimizer state likely not restored; '
          'investigate BEFORE the real run')

## 8. Train

Defaults: 20k steps, effective batch 64 (32x2 accum), checkpoint every 500 steps, fixed-seed 4x4 grid every **1000** steps (override with `--sample_every`). Grids cost ~12 min each on T4, so keep the cadence coarse - the grids, not the training, dominate wall-clock.

**Interrupted?** Just re-run this cell with the resume line below - optimizer/EMA/AMP state restores exactly.


In [ ]:
# fresh run - flags come from the CFG_FLAGS cell (0a), so 1c validated exactly
# this config and 1d projected exactly this cost. --sample_every 1000: each
# fixed-seed grid costs ~12 min on T4; the default 200 would spend ~1.5-2h of a
# 20k-step run on grids. 1000 -> ~21 grids (~4h).
F = CFG_FLAGS
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train.py  \
    --batch_size {F['batch_size']} --grad_accum_steps {F['grad_accum_steps']} --sample_every {F['sample_every']}  \
    --checkpoint_dir "$DRIVE_CKPT" --samples_dir "$DRIVE_SAMPLES" --log_csv "$DRIVE_LOG"

# ...or resume after an interruption (keep the three numeric flags in sync with 0a):
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train.py  \
#     --batch_size 32 --grad_accum_steps 2 --sample_every 1000  \
#     --resume_from "$DRIVE_CKPT/latest.pt"  \
#     --checkpoint_dir "$DRIVE_CKPT" --samples_dir "$DRIVE_SAMPLES" --log_csv "$DRIVE_LOG"

## 9. Watch progress

Sample grids use a FIXED seed: every frame denoises the same noise, so you literally watch sprites emerge.


In [ ]:
!python make_gif.py --samples_dir "$DRIVE_SAMPLES"
from IPython.display import Image
Image(filename=f'{DRIVE_SAMPLES}/training_progress.gif')

## 10. Inference from the latest checkpoint

Uses EMA weights; 16 sprites via the full 1000-step DDPM sampler (~4 min on T4 at batch 16).


In [ ]:
!python sample.py --checkpoint "$DRIVE_CKPT/latest.pt" --n 16 --nrow 4 --save_individual
from IPython.display import Image
import glob
Image(filename=sorted(glob.glob('samples/infer_step*.png'))[-1])

## Notes
- Effective batch is 32x2 = 64. Batch 64 in a single pass does NOT fit on a T4 (heuristic estimate ~14.1GB vs ~12.4GB safe fraction of the ~14.6GiB usable) — the guard will warn; expect OOM.
- Loss CSV lives at the `$DRIVE_LOG` path; plot with `pandas` if you want curves.
- Colab free tier disconnects after idle timeouts — the resume line in cell 7 makes that a non-event.
- Same checkpoint format as the local RTX 3050 run: `latest.pt` from Colab resumes locally and vice versa.